##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
from skimage import measure, filters
import statsmodels.api as sm
import matplotlib.pyplot as plt
import json

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def fit_lines(left_con_cand_pts,right_con_cand_pts):
    """
    Fits linear models to left and right ventricular contour points and calculates 
    the Splenial Angle.

    This function extracts the (x, y) coordinates from candidate contour points, 
    performs an Ordinary Least Squares (OLS) linear regression independently for 
    both the left and right borders to find their slopes, and then computes a 
    final geometric angle using their arctangents.

    Parameters
    ----------
    left_con_cand_pts : list of tuple of (int, int)
        A list of coordinate pairs `(x, y)` representing the candidate points 
        of the left ventricular contour boundary.
    right_con_cand_pts : list of tuple of (int, int)
        A list of coordinate pairs `(x, y)` representing the candidate points 
        of the right ventricular contour boundary.

    Returns
    -------
    angle : float
        The calculated splenial angle in degrees.
    """
    # Fit line to left ventricle contour points
    x_left, y_left = zip(*left_con_cand_pts)
    x_left = sm.add_constant(list(x_left))
    result_left = sm.OLS(y_left, x_left).fit()
    left_ven_slope = result_left.params[1]

    # Fit line to left ventricle contour points
    x_right, y_right = zip(*right_con_cand_pts)
    x_right = sm.add_constant(list(x_right))
    result_right = sm.OLS(y_right, x_right).fit()
    right_ven_slope = result_right.params[1]

    # Calculate and return the final splenial angle
    return 180 - abs(np.degrees(np.arctan(right_ven_slope))) - abs(np.degrees(np.arctan(left_ven_slope)))

In [ ]:
def rotate_image(image, im_type, dimension = 3):
    """Rotates axial and sagittal volumes for upright visualization.
    
    Expects the input volume to have a direction cosine of [-1,0,0,0,-1,0,0,0,1]. This is due to the nature of our preprocessing pipeline in 
    3D-Slicer (RAS coordinates) which involves a step of cropping the volume that sets direction cosines to be identity regardless of whether it was
    an axial/sagittal/coronal volume. When processing using SITK like in this notebook and plotting the volume with matplotlib, they appear inverted
    because SITK works in LPS coordinates. This function corrects rotations for axial and sagittal volumes. 
    
    Parameters
    ----------
    image : SimpleITK.Image
        The 3D image volume to be rotated/corrected.
    im_type : str
        The anatomical plane of the image. Expected values are 'Axial' or 
        'Sagittal' (or other strings, which default to the sagittal transformation).
    dimension : int, optional
        The dimensionality of the image transform (default is 3).

    Returns
    -------
    rotated_image : SimpleITK.Image
        The new SimpleITK image volume, resampled with the corrected 
        orientation and identity direction cosines.
    """
    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    
    if im_type == 'Axial':
        matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])
    else:
        matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,-1.0]])

    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def elliptical_mask(cen, a, b, shape, angle):
    """
    Generates a 2D binary mask containing a rotated ellipse.

    This function calculates an algebraic distance for every pixel in a grid 
    relative to a specified center, semi-axes lengths, and rotation angle. 
    Pixels falling inside or on the boundary of the ellipse are set to 1.

    Parameters
    ----------
    cen : tuple or list of (int, int)
        The center coordinates of the ellipse corresponding to (row, column).
    a : float or int
        The length of the semi-major axis (aligned with the row dimension at 0 rotation).
    b : float or int
        The length of the semi-minor axis (aligned with the column dimension at 0 rotation).
    shape : tuple of (int, int)
        The dimensions of the output binary mask corresponding to (height, width).
    angle : float
        The rotation angle of the ellipse in radians, evaluated counter-clockwise.

    Returns
    -------
    mask : numpy.ndarray of float64
        A 2D binary image array of the specified `shape` where 1 represents 
        the interior of the ellipse and 0 represents the background.
    """
    rows, cols = np.ogrid[:shape[0], :shape[1]]
    
    # Shift coordinates by the center points
    r_shifted = rows - int(cen[0])
    c_shifted = cols - int(cen[1])
    
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    
    # Calculate the rotated ellipse equation across the entire matrix at once
    term1 = (r_shifted * cos_a + c_shifted * sin_a) ** 2 / a ** 2
    term2 = (r_shifted * sin_a - c_shifted * cos_a) ** 2 / b ** 2
    
    return (term1 + term2 <= 1).astype(np.float64)

In [ ]:
def determine_start_ax_plane_via_3v_seg(sag_img_path, ax_img_path, pc, start_ax_plane_processing_params, visualize=False):
    """
    Determines the optimal starting axial plane for p-SA measurement by segmenting 
    the 3rd ventricle (3V) on the Mid-Sagittal Plane (MSP).

    The function uses the Posterior Commissure (PC) coordinates to isolate the MSP. 
    It applies Gaussian blurring and adaptive thresholding to segment the ventricles, 
    filters the results using an elliptical mask and blob analysis. To find the 
    starting coordinate, it identifies the superior-most boundary of the segmentation, 
    isolates its posterior-most column, and adds a slight inferior buffer. This 2D coordinate is then mapped back 
    into 3D physical space to find the corresponding axial slice index.

    Parameters
    ----------
    sag_img_path : str
        The path to the aligned sagittal volume
    ax_img_path : str
        The path to the aligned axial volume
    pc : tuple or list
        The (x, y, z) physical coordinates of the Posterior Commissure.
    start_ax_plane_processing_params : dict
        A dictionary containing predetermined ROI placement, morphological filtering, and intensity thresholding parameters.
    visualize : bool, optional
        If True, displays intermediate processing steps via matplotlib (default is False).

    Returns
    -------
    start_ax_plane : int
        The slice index of the axial volume corresponding to the optimal starting plane.
    """

    #isolate needed params
    brain_window_center = start_ax_plane_processing_params["brain_window_center"]
    brain_window_width = start_ax_plane_processing_params["brain_window_width"]
    csf_window_center = start_ax_plane_processing_params["csf_window_center"]
    csf_window_width = start_ax_plane_processing_params["csf_window_width"]
    third_ven_seg_initial_gauss_blur_size = start_ax_plane_processing_params["third_ven_seg_initial_gauss_blur_size"]
    msp_adaptive_thresh_size = start_ax_plane_processing_params["msp_adaptive_thresh_size"]
    third_ven_cen_roi_row_offset = start_ax_plane_processing_params["third_ven_cen_roi_row_offset"]
    third_ven_cen_roi_col_offset = start_ax_plane_processing_params["third_ven_cen_roi_col_offset"]
    third_ven_roi_major_axis = start_ax_plane_processing_params["third_ven_roi_major_axis"]
    third_ven_roi_minor_axis = start_ax_plane_processing_params["third_ven_roi_minor_axis"]
    third_ven_roi_tilt_degrees = start_ax_plane_processing_params["third_ven_roi_tilt_degrees"]
    
    
    #Read image, correct orientation, and window brain and CSF
    sag_img = sitk.ReadImage(sag_img_path)
    sag_img_brain = window_stack_sitk(rotate_image(sag_img, 'Sagittal'), brain_window_center, brain_window_width)            
    sag_img_csf = window_stack_sitk(rotate_image(sag_img, 'Sagittal'), csf_window_center, csf_window_width)
    
    #Using sagittal image LV segmentation as starting point for optimal axial plane search 
    sag_img_brain_dat = sitk.GetArrayFromImage(sag_img_brain)
    [i, j, k] = sag_img_csf.TransformPhysicalPointToContinuousIndex(pc) #lateral coordinate of the PC
    msp_image = sag_img_brain_dat[:, :, int(np.round(i))]
    
    #Add Guassian blur
    msp_gauss_blurred = cv.GaussianBlur(np.uint8(msp_image), (third_ven_seg_initial_gauss_blur_size,third_ven_seg_initial_gauss_blur_size),
                                        cv.BORDER_DEFAULT) 
    otsu_thresh = filters.threshold_otsu(msp_gauss_blurred)
    msp_bin = np.where(msp_gauss_blurred > otsu_thresh, 1, 0)
    
    #Adaptive thresholding to segment the ventricles 
    sag_adapt_thresh_img = cv.adaptiveThreshold(msp_gauss_blurred,1,cv.ADAPTIVE_THRESH_MEAN_C,
                                cv.THRESH_BINARY,msp_adaptive_thresh_size,1)
    
    #Define tilted sagittal ROI 
    row, col = msp_bin.nonzero()
    # Failsafe for empty mask
    if len(row) == 0:
        print(f"Warning: msp_bin is empty for {acc_num}. Returning default plane.")
        return 0 # Or whatever your default fallback plane should be

    cen = (min(row) + max(row))//2-third_ven_cen_roi_row_offset, (min(col) + max(col))//2-third_ven_cen_roi_col_offset 
    sag_c_mask = 1-elliptical_mask(cen, third_ven_roi_major_axis, third_ven_roi_minor_axis, msp_bin.shape,
                                   np.radians(third_ven_roi_tilt_degrees)) 
    
    #This captures the top part of the LV 
    non_lv_sag = np.logical_or(sag_adapt_thresh_img, sag_c_mask)
    seg_area = 1-non_lv_sag
    
    
    #If there are other elements which are captured, it checks for the horizontal range and keeps the one with the largest y-range
    #This check also captures the vertical range to eliminate false positives that are prominent sulci or atrophy 
    blobs_labels = measure.label(seg_area, background = 0)
    largest_2_nb_blobs = np.argsort(np.bincount(blobs_labels.flat))[::-1][1:3] #excluding the background, grab the top 2 largest blob arguments  
    
    #Ensure blobs were actually found before indexing
    if len(largest_2_nb_blobs) == 0:
        print(f"Warning: No valid blobs found in {acc_num}.")
        return None 
        
    f_blob = blobs_labels == largest_2_nb_blobs[0]
    sag_ven = f_blob
    
    if len(largest_2_nb_blobs) > 1: 
        
        [rf, cf] = f_blob.nonzero()
        f_blob_range = max(cf) - min(cf)
        cf_s = sorted(np.unique(cf))
        f_blob_vert_range = []
        for cf_ss in cf_s:
            f_blob_vert_range = f_blob_vert_range + [max(rf[cf==cf_ss]) - min(rf[cf==cf_ss])]
        f_blob_vert_range = np.median(f_blob_vert_range)
    
        s_blob = blobs_labels == largest_2_nb_blobs[1]
        [rs, cs] = s_blob.nonzero()
        s_blob_range = max(cs) - min(cs)
        cs_s = sorted(np.unique(cs))
        s_blob_vert_range = []
        for cs_ss in cs_s:
            s_blob_vert_range = s_blob_vert_range + [max(rs[cs==cs_ss]) - min(rs[cs==cs_ss])]
        s_blob_vert_range = np.median(s_blob_vert_range)
    
        #If the smaller blob spans a larger vertical and horizontal range than the first larger blob, the small blob is picked up
        if (f_blob_range < s_blob_range) and (f_blob_vert_range < s_blob_vert_range):
            print('small blob picked up')
            sag_ven = s_blob
    
    row, col = sag_ven.nonzero()
    
    #Ensure the final segmented mask is not empty before applying min/max
    if len(row) == 0:
        print(f"Warning: sag_ven mask is empty for {acc_num}")
        return None 

    mcs = max(col[row == min(row)]) #defines the right-most (posterior-most) column corresponding to the superior-most row the segmented 3V
    mrs = min(row[col == mcs]) + 5 #defines the top-most row corresponding to mcs and pushes it 5 mm inferiorly for a buffer 

    #Aligned axial image for p-SA measurement
    ax_img = sitk.ReadImage(ax_img_path)     
    #Read image, correct orientation, and window brain
    ax_img_brain = window_stack_sitk(rotate_image(ax_img,'Axial'), brain_window_center, brain_window_width)
    
    #transform the z-coordinate identified previously (mrs that is the voxel coordinate to physical coordinates and back to 
    #image coordinates corresponding to the axial volume) 
    start_ax_plane = int(np.round(ax_img_brain.TransformPhysicalPointToContinuousIndex(
                                  sag_img_brain.TransformContinuousIndexToPhysicalPoint([int(np.round(i)), 
                                                                                         int(np.round(mcs)), 
                                                                                         msp_image.shape[0] - int(np.round(mrs))
                                                                                        ]))[2]))
    print(start_ax_plane)
    
    if visualize:
        plt.imshow(msp_image,cmap = 'gray')
        plt.scatter(np.round(j),msp_image.shape[0]-np.round(k),marker = 'x')
        plt.title('MSP image with PC visible')
        plt.show()                    
        plt.imshow(msp_gauss_blurred,cmap = 'gray')
        plt.title('MSP blurred with Gaussian')
        plt.show()
        plt.imshow(sag_adapt_thresh_img, cmap = 'gray')
        plt.title('Adaptive threshold')
        plt.show()        
        plt.imshow(sag_ven,cmap = 'gray')
        plt.title('Segmented LV')
        plt.show()
    
    return start_ax_plane, msp_image, seg_area, mcs, mrs

In [ ]:
def determine_opt_ax_plane_for_psa_via_wm_profile(start_ax_plane, ax_img_path, msp_image, seg_area, mcs, mrs, opt_ax_plane_processing_params, 
                                                  visualize=False):
    """
    Determines the optimal axial plane for p-SA measurement by analyzing the 
    white matter (WM) profile within the Corpus Callosum (CC).

    Starting from a baseline axial slice, this function segments the lateral 
    ventricles and defines an ROI spanning the corpus callosum. It then iterates 
    superiorly across a defined number of slices, quantifying the white matter 
    pixels within the ROI. The slice with the peak WM profile is selected as 
    the optimal plane.

    Parameters
    ----------
    start_ax_plane : int
        The slice index of the starting axial plane.
    ax_img_path : str
        Path to AC-PC aligned axial CT. 
    opt_ax_plane_processing_params : dict
        Dictionary of parameters (thresholds, kernel sizes, morphological offsets).
    visualize : bool, optional
        If True, displays intermediate processing steps via matplotlib (default is False).
    msp_image, seg_area, mcs, mrs : various, optional
        Data from the prior 3V segmentation step. Passed solely to generate the 
        final multi-panel visualization if requested.

    Returns
    -------
    opt_plane : int
        The slice index corresponding to the optimal axial plane.
    """

    ax_adaptive_thresh_size = opt_ax_plane_processing_params["ax_adaptive_thresh_size"]
    ax_lat_ven_roi_semi_major_axis = opt_ax_plane_processing_params["ax_lat_ven_roi_semi_major_axis"]
    ax_lat_ven_roi_semi_minor_axis = opt_ax_plane_processing_params["ax_lat_ven_roi_semi_minor_axis"]
    lv_roi_close_kernel_size = opt_ax_plane_processing_params["lv_roi_close_kernel_size"]
    num_close_iters_lv_roi_ax = opt_ax_plane_processing_params["num_close_iters_lv_roi_ax"]
    max_superior_search_ax_planes = opt_ax_plane_processing_params["max_superior_search_ax_planes"]
    wm_low_thresh = opt_ax_plane_processing_params["wm_low_thresh"]
    wm_high_thresh = opt_ax_plane_processing_params["wm_high_thresh"]
    cc_roi_col_offset = opt_ax_plane_processing_params["cc_roi_col_offset"]
    ax_plane_gauss_blur_kernel_size = opt_ax_plane_processing_params["ax_plane_gauss_blur_kernel_size"]
    brain_window_center = opt_ax_plane_processing_params["brain_window_center"]
    brain_window_width = opt_ax_plane_processing_params["brain_window_width"]
    
    #Read image, correct orientation, and window brain
    # print(ax_img_path)
    ax_img = sitk.ReadImage(ax_img_path)     
    ax_img_brain = window_stack_sitk(rotate_image(ax_img,'Axial'), brain_window_center, brain_window_width)
    ax_img_brain_arr = sitk.GetArrayFromImage(ax_img_brain)

    # print(ax_img_brain_arr.shape), print(start_ax_plane)
    
    ax_plane_sa = ax_img_brain_arr[start_ax_plane,:,:]
    otsu_thresh = filters.threshold_otsu(ax_plane_sa)
    ax_plane_sa_bin = ax_plane_sa > otsu_thresh
    #Add Guassian blur
    ax_gauss_blurred = cv.GaussianBlur(np.uint8(ax_plane_sa),(3,3), cv.BORDER_DEFAULT) 
    #Adaptive thresholding to segment the ventricles 
    ax_adapt_thresh_img = cv.adaptiveThreshold(ax_gauss_blurred,1,cv.ADAPTIVE_THRESH_MEAN_C,
                                cv.THRESH_BINARY,ax_adaptive_thresh_size,1) 
    
    M = cv.moments(np.uint8(ax_plane_sa_bin))
    
    if M["m00"] != 0:
        mid_col = int(M["m10"] / M["m00"])
        mid_row = int(M["m01"] / M["m00"])
    else:
        mid_col, mid_row = ax_plane_sa_bin.shape[1]//2, ax_plane_sa_bin.shape[0]//2


    around_ventricle_mask = 1-elliptical_mask((mid_row, mid_col), ax_lat_ven_roi_semi_major_axis,
                                              ax_lat_ven_roi_semi_minor_axis, ax_plane_sa_bin.shape, 0) 
    
    #Close around the ventricles
    close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (lv_roi_close_kernel_size,lv_roi_close_kernel_size))

    closed_ax_brain_mask = cv.morphologyEx(np.uint8(ax_adapt_thresh_img*around_ventricle_mask), 
                                                    cv.MORPH_CLOSE, close_kernel, 
                                                    iterations = num_close_iters_lv_roi_ax)

    #Get mask covering the lateral ventricles in the center
    ven_sur_mask = (1-closed_ax_brain_mask)*(1-ax_adapt_thresh_img)
    blobs_labels = measure.label(ven_sur_mask, background = 0)
    

    background_pixels = getLargestCC(blobs_labels)
    ven_sur_mask = np.where(background_pixels > 0,0,ven_sur_mask)

    if visualize:
        plt.imshow(ax_adapt_thresh_img, cmap = 'gray')
        plt.title('Adaptive threshold around starting axial plane')
        plt.show()
        plt.imshow(closed_ax_brain_mask,cmap = 'gray')
        plt.title('Closed around ventricles mask')
        plt.show()              
        plt.imshow(background_pixels,cmap = 'gray')
        plt.title('Background pixels')
        plt.show()
        plt.imshow(ven_sur_mask,cmap = 'gray')
        plt.title('Mask covering LV and area around them')
        plt.show()  

    #define upper and lower points on the CC ROI   
    [nz_rv, nz_cv] = ven_sur_mask.nonzero()
    M_ven = cv.moments(np.uint8(ven_sur_mask))
    

    if M_ven["m00"] != 0:
        mc = int(M_ven["m10"] / M_ven["m00"])
    else:
        mc = mid_col 
    
    ##starting at the middle column, find the top-most and bottom-most segmented points on the LVs. 
    ##keep moving left until non-zero pixels are found
    i = 0
    lr, ur = 0, ven_sur_mask.shape[0] - 1 # Safe defaults
    
    while (mc - i) >= 0:
        try:
            lr = min(nz_rv[nz_cv == mc-i])
            break
        except ValueError:
            i += 1
            
    i = 0
    while (mc - i) >= 0:
        try:
            ur = max(nz_rv[nz_cv == mc-i])
            break
        except ValueError:
            i += 1
            
    #if the top-most and bottom-most segmented points overlap, move left until distinct points are found
    while lr == ur and (mc - i) >= 0:
        i += 1
        try:
            lr = min(nz_rv[nz_cv == mc-i])
            ur = max(nz_rv[nz_cv == mc-i])
        except ValueError:
            pass

    if visualize:
        plt.imshow(ven_sur_mask,cmap = 'gray')
        plt.scatter([x for x in range(ven_sur_mask.shape[1])], [lr]*ven_sur_mask.shape[1], marker = 'x',s=5)
        plt.scatter([x for x in range(ven_sur_mask.shape[1])], [ur]*ven_sur_mask.shape[1], marker = 'x',s=5)
        # Ensure x-coordinates stay >= 0 for visualization
        safe_left_bound = max(0, mc-cc_roi_col_offset)
        plt.scatter([safe_left_bound]*ven_sur_mask.shape[0],[x for x in range(ven_sur_mask.shape[0])], marker = 'x',s=5)
        plt.scatter([mc+cc_roi_col_offset]*ven_sur_mask.shape[0],[x for x in range(ven_sur_mask.shape[0])], marker = 'x',s=5)
        plt.title('Finding the CC mask boundary pts.')
        plt.show()

    cc_body_mask = np.zeros(ven_sur_mask.shape)
    safe_left_bound = max(0, mc-cc_roi_col_offset)
    cc_body_mask[lr:ur, safe_left_bound:mc+cc_roi_col_offset] = 1

    #Gather the WM profile within the CC ROI starting from the predetermined starting plane, up to 15 mm superiorly
    cc_sum_arr = []
    for pl in range(start_ax_plane, start_ax_plane + max_superior_search_ax_planes): 
        # Safely bound the plane index to the image array
        if pl >= ax_img_brain_arr.shape[0]:
            break
            
        ax_plane_sa = ax_img_brain_arr[pl,:,:]
        
        ax_plane_sa_gauss_blurred = cv.GaussianBlur(np.uint8(ax_plane_sa),(ax_plane_gauss_blur_kernel_size,
                                                                  ax_plane_gauss_blur_kernel_size), cv.BORDER_DEFAULT)

        ax_img_wm = ax_plane_sa_gauss_blurred.copy()
        ax_img_wm[(ax_plane_sa_gauss_blurred < wm_low_thresh)|(ax_plane_sa_gauss_blurred > wm_high_thresh)] = 0            
        
        cc_body = ax_img_wm*cc_body_mask        
        cc_sum_arr = cc_sum_arr + [np.sum(cc_body)]

    #normalize the WM profile in the CC ROI and determine the axial slice where this value peaks
    if len(cc_sum_arr) > 0 and max(cc_sum_arr) > 0:
        cc_sum_arr_norm = cc_sum_arr/(max(cc_sum_arr))
        opt_plane = start_ax_plane + np.argmax(cc_sum_arr_norm)
    else:
        print("Warning: CC WM profile array was empty or zero. Returning starting plane.")
        opt_plane = start_ax_plane
        cc_sum_arr_norm = [0] * max_superior_search_ax_planes # Dummy for plotting
    

    fig, axes = plt.subplots(1, 3, figsize=(24, 6))


    if all(v is not None for v in [msp_image, seg_area, mcs, mrs]):
        axes[0].imshow(msp_image, cmap='gray')
        axes[0].imshow(seg_area, alpha=0.5)
        axes[0].scatter(mcs, mrs - max_superior_search_ax_planes, marker='x', color='white')
        axes[0].scatter(mcs, mrs, marker='x', color='white')
        axes[0].set_title('Slices examined')        
    else:
        axes[0].text(0.5, 0.5, 'MSP data not provided\nSkipping plot', horizontalalignment='center', verticalalignment='center')
        axes[0].set_title('Slices examined (No Data)')
        axes[0].axis('off')
        
    # Generate an x-axis range that matches the length of the found cc_sum_arr_norm 
    # (in case the loop broke early due to image boundaries)
    x_range = range(start_ax_plane, start_ax_plane + len(cc_sum_arr_norm))
    axes[1].scatter(list(x_range), cc_sum_arr_norm, color='black')
    axes[1].set_title('CC pixels profile')      
    axes[2].imshow(ax_img_brain_arr[opt_plane, :, :], cmap='gray')
    axes[2].set_title(f'Axial plane for SA measurement: {opt_plane}')
    
    plt.show() 

    return opt_plane

In [ ]:
def trace_medial_walls_proxy_for_forceps(processed_ventricles, ax_plane_sa, clr):
    """
    Traces contour points on the medial walls of the occipital horns on the segmented lateral ventricles.

    This function isolates the left and right ventricles, identifies starting points 
    at the medial bounds, and mathematically "walks" down the medial boundaries 
    (South, South-West/South-East, or West/East) to extract a contour line. The points 
    are then filtered via `pt_selector` and visualized on the original axial plane.

    Parameters
    ----------
    processed_ventricles : numpy.ndarray
        A 2D binary mask (image) containing the segmented lateral ventricles.
    ax_plane_sa : numpy.ndarray
        The optimal 2D axial slice (image) used for plotting the final scatter points.
    clr : str
        The color to use for matplotlib scatter points (e.g., 'r' or 'blue').

    Returns
    -------
    left_con_pts : list of tuples
        Filtered (x, y) coordinates for the medial wall of the left ventricle.
    right_con_pts : list of tuples
        Filtered (x, y) coordinates for the medial wall of the right ventricle.
    """
    
    [nr, nc] = processed_ventricles.nonzero()
    # Safety check: if mask is entirely empty, return empty lists
    if len(nr) == 0:
        print("Warning: Ventricle mask is empty.")
        return [], []

    mid_col = (min(nc) + max(nc))//2

    left_ven = processed_ventricles[:,:mid_col]
    left_ven = getLargestCC(measure.label(left_ven, background=0))

    right_ven = processed_ventricles[:,mid_col:]
    right_ven = getLargestCC(measure.label(right_ven, background=0))

    # Image boundaries to prevent Out of Bounds errors
    max_row_bound, max_col_bound = left_ven.shape

    [lnr, lnc] = left_ven.nonzero()

    try:
        l_term_row, l_term_col = max(lnr)-2, max(lnc[lnr == max(lnr)-2])
    except ValueError:
        l_term_row, l_term_col = max(lnr), max(lnc[lnr == max(lnr)])
    l_row, l_col = min(lnr), max(lnc[lnr == min(lnr)])

    [rnr, rnc] = right_ven.nonzero()
    try:
        r_term_row, r_term_col = max(rnr)-2, min(rnc[rnr == max(rnr)-2])
    except ValueError:
        r_term_row, r_term_col = max(rnr), min(rnc[rnr == max(rnr)])
    r_row, r_col = min(rnr), min(rnc[rnr == min(rnr)])

    left_pts = []
    while not((l_row >= l_term_row) and (l_col <= l_term_col)):    
        if l_row + 1 >= max_row_bound or l_col - 1 < 0:
            break
            
        if left_ven[l_row+1, l_col] == 1: #south 
            left_pts.append([l_col,l_row+1])
            l_row, l_col = l_row+1, l_col
        elif left_ven[l_row+1, l_col-1] == 1: #south-west
            left_pts.append([l_col-1,l_row+1])
            l_row, l_col = l_row+1, l_col-1
        else: 
            left_pts.append([l_col-1,l_row]) #default west
            l_row, l_col = l_row, l_col-1

    right_pts = []

    # Image boundaries to prevent Out of Bounds errors
    rmax_row_bound, rmax_col_bound = right_ven.shape
    
    while not((r_row >= r_term_row) and (r_col >= r_term_col)):    
        if r_row + 1 >= rmax_row_bound or r_col + 1 >= rmax_col_bound:
            break
            
        if right_ven[r_row+1, r_col] == 1: #south 
            right_pts.append([r_col,r_row+1])
            r_row, r_col = r_row+1, r_col
        elif right_ven[r_row+1, r_col+1] == 1: #south-east
            right_pts.append([r_col+1,r_row+1])
            r_row, r_col = r_row+1, r_col+1
        else: 
            right_pts.append([r_col+1,r_row]) #default east
            r_row, r_col = r_row, r_col+1    
            
    right_pts = [(x+mid_col, y) for x,y in right_pts]

    left_con_pts, right_con_pts = pt_selector(left_pts, right_pts)

    fig, ax = plt.subplots(figsize=(10,10))
    ax.imshow(ax_plane_sa, cmap='gray')

    [ax.scatter(x, y, marker='x', color=clr, s=10) for x,y in left_con_pts]
    [ax.scatter(x, y, marker='x', color=clr, s=10) for x,y in right_con_pts]
    plt.show()
    
    return left_con_pts, right_con_pts

In [ ]:
def pt_selector(left_pts, right_pts, prcnt_upper_x=0.05, prcnt_lower_y=0.05):
    """
    Filters and trims contour points to isolate salient points on the medial walls of the occipital horns.

    This function first isolates the south-west/west direction for the left ventricular points 
    and the south-east/east direction for the right ventricular points by analyzing coordinate 
    gradients. It then adaptively trims the extreme top, bottom, left, and right 
    tails of the contours by a specified percentage threshold.

    Parameters
    ----------
    left_pts : list of tuples or list of lists
        Coordinates (x, y) making up the left trace/contour.
    right_pts : list of tuples or list of lists
        Coordinates (x, y) making up the right trace/contour.
    prcnt_upper_x : float, optional
        The percentage of the total X-axis range to trim from the left and right 
        edges of the contours (default is 0.05, representing 5%).
    prcnt_lower_y : float, optional
        The percentage of the total Y-axis range to trim from the top and bottom 
        edges of the contours (default is 0.05, representing 5%).

    Returns
    -------
    left_con_pts : list of tuples
        The filtered and trimmed coordinates for the left contour.
    right_con_pts : list of tuples
        The filtered and trimmed coordinates for the right contour.
    """

    #grab the left points where the x-coordinate or column first decreases up to the last such point -- basically ensuring that south-west or west
    #directions are being captured in the very beginning and very end of this contour
    left_con_pts = left_pts[np.where(np.diff([x for x,y in left_pts])<0)[0][0]:
                            np.where(np.diff([x for x,y in left_pts])<0)[0][-1]+1]
    #grab the indices where the x coordinate decreases AND where the y coordinate or row increases
    left_selected_indices = np.where(np.array([x for x in (np.diff([x for x,y in left_con_pts]) < 0)] + [True]) & 
                                     np.array([x for x in (np.diff([y for x,y in left_con_pts]) > 0)] + [True]))
    left_con_pts = [left_con_pts[li] for li in left_selected_indices[0]]

    #grab the right points where the x-coordinate or column first increases up to the last such point -- basically ensuring that south-east or east
    #directions are being captured in the very beginning and very end of this contour
    right_con_pts = right_pts[np.where(np.diff([x for x,y in right_pts])>0)[0][0]:
                            np.where(np.diff([x for x,y in right_pts])>0)[0][-1]+1]
    #grab the indices where the x coordinate increases AND where the y coordinate or row increases
    right_selected_indices = np.where(np.array([x for x in (np.diff([x for x,y in right_con_pts]) > 0)] + [True]) & 
            np.array([x for x in (np.diff([y for x,y in right_con_pts]) > 0)] + [True]))
    right_con_pts = [right_con_pts[li] for li in right_selected_indices[0]]


    # ==========================================
    # LEFT CONTOUR - Y-AXIS TRIMMING
    # ==========================================
    left_y_range = left_con_pts[-1][1] - left_con_pts[0][1]
    if left_y_range > 5:
        # Initialize safe defaults to prevent UnboundLocalError
        first_l_pt_y = 0
        last_l_pt_y = len(left_con_pts)
        
        ly1, y_thresh,i = left_con_pts[-1][1], (left_y_range)*prcnt_lower_y, 0
        while True:    
            l_grad_cond = np.where(np.array([ly1-ly for lx,ly in left_con_pts])>(y_thresh-i))
            if (i > y_thresh):
                print('Suitable left y_thresh not found')
                break
            if len(l_grad_cond[0]) > 0:
                last_l_pt_y = l_grad_cond[0][-1]
                break
            else:
                i = i + 1
        ly0, i = left_con_pts[0][1], 0
        while True:
            l_grad_cond = np.where(np.array([ly-ly0 for lx,ly in left_con_pts])>(y_thresh-i))
            if (i > y_thresh):
                print('Suitable left y_thresh not found')
                break
            if len(l_grad_cond[0]) > 0:
                first_l_pt_y = l_grad_cond[0][0]
                break
            else:
                i = i + 1
        if len(left_con_pts[first_l_pt_y:last_l_pt_y]) > 5:
            left_con_pts = left_con_pts[first_l_pt_y:last_l_pt_y]
    else:
        left_con_pts = left_con_pts

    # ==========================================
    # LEFT CONTOUR - X-AXIS TRIMMING
    # ==========================================
    left_x_range = left_con_pts[0][0] - left_con_pts[-1][0]
    if left_x_range > 5:    
        # Initialize safe defaults to prevent UnboundLocalError
        first_l_pt_x = 0
        last_l_pt_x = len(left_con_pts)
        
        lx0,x_thresh,i = left_con_pts[0][0],(left_x_range)*prcnt_upper_x,0
        while True:    
            l_grad_cond = np.where(np.array([lx0-lx for lx,ly in left_con_pts])>(x_thresh-i))
            if (i > x_thresh):
                print('Suitable left x_thresh not found')
                break
            if len(l_grad_cond[0]) > 0:
                first_l_pt_x = l_grad_cond[0][0]
                break
            else:
                i = i + 1
        lx1,i = left_con_pts[-1][0],0
        while True:    
            l_grad_cond = np.where(np.array([lx-lx1 for lx,ly in left_con_pts])>(x_thresh-i))
            if (i > x_thresh):
                print('Suitable left x_thresh not found')
                break
            if len(l_grad_cond[0]) > 0:
                last_l_pt_x = l_grad_cond[0][-1]
                break
            else:
                i = i + 1
        if len(left_con_pts[first_l_pt_x:last_l_pt_x]) > 5:
            left_con_pts = left_con_pts[first_l_pt_x:last_l_pt_x]
    else:
        left_con_pts = left_con_pts

    # ==========================================
    # RIGHT CONTOUR - Y-AXIS TRIMMING
    # ==========================================
    right_y_range = right_con_pts[-1][1] - right_con_pts[0][1]   
    if right_y_range > 5:
        # Initialize safe defaults to prevent UnboundLocalError
        first_r_pt_y = 0
        last_r_pt_y = len(right_con_pts)
        
        ry1, y_thresh,i = right_con_pts[-1][1], (right_y_range)*prcnt_lower_y,0
        while True:    
            r_grad_cond = np.where(np.array([ry1-ry for rx,ry in right_con_pts])>(y_thresh-i))
            if (i > y_thresh):
                print('Suitable right y_thresh not found')
                break
            if len(r_grad_cond[0]) > 0:
                last_r_pt_y = r_grad_cond[0][-1]
                break
            else:
                i = i + 1   
        ry0, i = right_con_pts[0][1],0
        while True:    
            r_grad_cond = np.where(np.array([ry-ry0 for rx,ry in right_con_pts])>(y_thresh-i))
            if (i > y_thresh):
                print('Suitable right y_thresh not found')
                break
            if len(r_grad_cond[0]) > 0:
                first_r_pt_y = r_grad_cond[0][0]
                break
            else:
                i = i + 1
        if len(right_con_pts[first_r_pt_y:last_r_pt_y]) > 5:
            right_con_pts = right_con_pts[first_r_pt_y:last_r_pt_y]
    else:
        right_con_pts = right_con_pts

    # ==========================================
    # RIGHT CONTOUR - X-AXIS TRIMMING
    # ==========================================
    right_x_range = right_con_pts[-1][0] - right_con_pts[0][0]
    if right_x_range > 5:
        # Initialize safe defaults to prevent UnboundLocalError
        first_r_pt_x = 0
        last_r_pt_x = len(right_con_pts)
        
        rx0,x_thresh,i = right_con_pts[0][0],(right_x_range)*prcnt_upper_x,0
        while True:    
            r_grad_cond = np.where(np.array([rx-rx0 for rx,ry in right_con_pts])>(x_thresh-i))
            if (i > x_thresh):
                print('Suitable right x_thresh not found')
                break
            if len(r_grad_cond[0]) > 0:
                first_r_pt_x = r_grad_cond[0][0]
                break
            else:
                i = i + 1
        rx1, i = right_con_pts[-1][0],0
        while True:    
            r_grad_cond = np.where(np.array([rx1-rx for rx,ry in right_con_pts])>(x_thresh-i))
            if (i > x_thresh):
                print('Suitable right x_thresh not found')
                break
            if len(r_grad_cond[0]) > 0:
                last_r_pt_x = r_grad_cond[0][-1]
                break
            else:
                i = i + 1
        if len(right_con_pts[first_r_pt_x:last_r_pt_x]) > 5:
            right_con_pts = right_con_pts[first_r_pt_x:last_r_pt_x]
    else: 
        right_con_pts = right_con_pts


    return left_con_pts, right_con_pts

In [ ]:
def segment_occipital_horns_fit_lines(ax_img_path, opt_plane, start_ax_plane, occipital_horns_segment_params):
    """
    Segments the occipital horns of the lateral ventricles and calculates the 
    proxy splenial angle (p-SA) by fitting lines along their medial walls.

    This function iterates through axial planes starting from the optimal plane, 
    moving inferiorly if valid ventricular segmentations are not found. It uses 
    flood-filling, adaptive thresholding, and morphological operations to isolate 
    the ventricles. It then splits the mask into left and right hemispheres to 
    trace the occipital horns (approximate contour of the forceps major) and calculates the resultant angle.

    Parameters
    ----------
    ax_img_brain_arr : numpy.ndarray
        3D numpy array of the brain-windowed axial CT.
    opt_plane : int
        The slice index of the optimal axial plane determined in the previous step.
    start_ax_plane : int
        The slice index of the baseline axial plane. Serves as the inferior bound 
        for the slice search.
    occipital_horns_segment_params : dict
        Dictionary of parameters (blur sizes, kernels, thresholds, etc.).
    

    Returns
    -------
    p_sa_angle : float or None
        The calculated proxy splenial angle. Returns None if the algorithm fails to 
        find valid segmentations above or at the starting axial plane.
    """

    lat_ven_seg_initial_gauss_blur_size = occipital_horns_segment_params["lat_ven_seg_initial_gauss_blur_size"]
    lv_roi_close_kernel_size = occipital_horns_segment_params["lv_roi_close_kernel_size"]
    num_close_iters_roi_ax = occipital_horns_segment_params["num_close_iters_roi_ax"]
    ax_adaptive_thresh_size = occipital_horns_segment_params["ax_adaptive_thresh_size"]
    ven_seg_to_full_brain_ratio = occipital_horns_segment_params["ven_seg_to_full_brain_ratio"]
    bilateral_filter_size = occipital_horns_segment_params["bilateral_filter_size"]
    bilater_filter_sigma_color = occipital_horns_segment_params["bilater_filter_sigma_color"]
    bilater_filter_sigma_space = occipital_horns_segment_params["bilater_filter_sigma_space"]
    lv_middle_roi_col_offset = occipital_horns_segment_params["lv_middle_roi_col_offset"]
    lv_middle_roi_row_offset = occipital_horns_segment_params["lv_middle_roi_row_offset"]
    close_roi_left_prcnt_fbc = occipital_horns_segment_params["close_roi_left_prcnt_fbc"]
    close_roi_cen_exclude = occipital_horns_segment_params["close_roi_cen_exclude"]
    ax_lv_close_iter = occipital_horns_segment_params["ax_lv_close_iter"]
    ven_median_smooth_factor = occipital_horns_segment_params["ven_median_smooth_factor"]
    min_ven_area_thresh = occipital_horns_segment_params["min_ven_area_thresh"]
    brain_window_center = occipital_horns_segment_params["brain_window_center"]
    brain_window_width = occipital_horns_segment_params["brain_window_center"]


    #Read image, correct orientation, and window brain
    ax_img = sitk.ReadImage(ax_img_path)     
    ax_img_brain = window_stack_sitk(rotate_image(ax_img,'Axial'), brain_window_center, brain_window_width)
    ax_img_brain_arr = sitk.GetArrayFromImage(ax_img_brain)
    
    while True: 
        
        if opt_plane < start_ax_plane - 10:
            print(f"Warning: Dropped 10 mm below start_ax_plane ({start_ax_plane}). Returning None.")
            return None

        ax_plane_sa = ax_img_brain_arr[opt_plane,:,:]
        ax_gauss_blurred = cv.GaussianBlur(np.uint8(ax_plane_sa),(lat_ven_seg_initial_gauss_blur_size,lat_ven_seg_initial_gauss_blur_size),
                                           cv.BORDER_DEFAULT)  
        otsu_thresh = filters.threshold_otsu(ax_gauss_blurred)
        ax_plane_sa_bin = ax_gauss_blurred > otsu_thresh
        
        
        M = cv.moments(np.uint8(ax_plane_sa_bin))
        if M["m00"] != 0:
            mid_col = int(M["m10"] / M["m00"])
            mid_row = int(M["m01"] / M["m00"])
        else:
            mid_col, mid_row = ax_plane_sa_bin.shape[1]//2, ax_plane_sa_bin.shape[0]//2

        #Close around the ventricles
        close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (lv_roi_close_kernel_size,lv_roi_close_kernel_size))
        closed_ax_brain_mask = cv.morphologyEx(np.uint8(ax_plane_sa_bin), 
                                               cv.MORPH_CLOSE, close_kernel, 
                                               iterations = num_close_iters_roi_ax)
        im_th = closed_ax_brain_mask.copy()
        h, w = im_th.shape[:2]
        mask = np.zeros((h+2, w+2), np.uint8)
        im_floodfill = im_th.copy()
        
        # Floodfill from point (0, 0)
        res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)

        full_brain_mask = (1-res[2]).copy()
        full_brain_mask = full_brain_mask[1:h+1,1:w+1]
        
        
        [fbr, fbc] = full_brain_mask.nonzero()
        if len(fbc) == 0:
            opt_plane = opt_plane - 1
            continue
            
        fbc_range = max(fbc) - min(fbc)
        
        
        M_fb = cv.moments(np.uint8(full_brain_mask))
        if M_fb["m00"] != 0:
            bcX = int(M_fb["m10"] / M_fb["m00"])
            bcY = int(M_fb["m01"] / M_fb["m00"])
        else:
            bcX, bcY = full_brain_mask.shape[1]//2, full_brain_mask.shape[0]//2
            
        ax_plane_adapt_thresh = cv.adaptiveThreshold(ax_gauss_blurred,1,cv.ADAPTIVE_THRESH_MEAN_C,
                                        cv.THRESH_BINARY,ax_adaptive_thresh_size,1)
        
        crude_seg = (1-ax_plane_adapt_thresh)*(full_brain_mask)
            
        if (np.sum(crude_seg)/np.sum(full_brain_mask) > ven_seg_to_full_brain_ratio) or (opt_plane == start_ax_plane):     
            #loop executes and breaks if we are happy with ventricle size OR we are the bottom-most plane

            crude_seg_smooth = cv.bilateralFilter(np.uint8(crude_seg),
                                                  bilateral_filter_size,
                                                  bilater_filter_sigma_color,
                                                  bilater_filter_sigma_space)
            middle_roi = np.zeros(crude_seg_smooth.shape)
            mr_v, mc_v = crude_seg_smooth.shape[0]//2, crude_seg_smooth.shape[1]//2 
            middle_roi[bcY-lv_middle_roi_col_offset:bcY+lv_middle_roi_col_offset, 
                       bcX-lv_middle_roi_row_offset:bcX+lv_middle_roi_row_offset] = 1 
            
            clean_ven_1 = np.logical_or(middle_roi, crude_seg_smooth)
            blobs_labels = measure.label(clean_ven_1, background = 0)
            
           
            largest_cc_ven = getLargestCC(blobs_labels) 
            [row_v, col_v] = (1-largest_cc_ven).nonzero()
            
            ventricles_final = crude_seg_smooth.copy()
            ventricles_final[row_v, col_v] = 0
            close_roi = np.zeros(ventricles_final.shape)
            
            #want to avoid closing a thin strip of 0.1 mm in the center to avoid any atrophy/midline
            close_roi[:,bcX-int(np.round(fbc_range*close_roi_left_prcnt_fbc)):bcX-int(np.round(fbc_range*close_roi_cen_exclude/2))] = 1
            close_roi[:,bcX+int(np.round(fbc_range*close_roi_cen_exclude/2)):bcX+int(np.round(fbc_range*close_roi_left_prcnt_fbc))] = 1
            
            #emphasizing south-west and south-east connections
            sw_kernel_close = np.array([[1,1,0],[0,0,0],[0,1,1]],'uint8')
            se_kernel_close = np.array([[0,1,1],[0,0,0],[1,1,0]],'uint8')

            close_ventricles = cv.morphologyEx(cv.morphologyEx(np.uint8(close_roi*ventricles_final), 
                                                    cv.MORPH_CLOSE,
                                                     sw_kernel_close,
                                                    iterations = ax_lv_close_iter),
                                               cv.MORPH_CLOSE,
                                               se_kernel_close,
                                               iterations = ax_lv_close_iter)   
            close_ventricle_mask = np.logical_or(close_ventricles,ventricles_final*(1-close_roi))                 
            ventricles_smoothed = cv.medianBlur(np.uint8(close_ventricle_mask),ven_median_smooth_factor) 
            
            bot_ventricles = ventricles_smoothed.copy()
            bot_ventricles[:mid_row,:] = 0

            #fall back is to move inferior when segmentations are null. 
            if np.sum(bot_ventricles) == 0:
                opt_plane = opt_plane - 1
                continue
        
            processed_ventricles = bot_ventricles
            [nr, nc] = processed_ventricles.nonzero()
            mid_col = (max(nc) + min(nc))//2
            
            left_ven = processed_ventricles[:,:mid_col]
            right_ven = processed_ventricles[:,mid_col:]
            
            if ((np.sum(left_ven) <= min_ven_area_thresh) or (np.sum(right_ven) <= min_ven_area_thresh)) and (opt_plane > start_ax_plane):
                opt_plane = opt_plane - 1
                continue
                
            try:
                left_con_pts, right_con_pts = trace_medial_walls_proxy_for_forceps(processed_ventricles, ax_plane_sa, 'w')
                p_sa_angle = fit_lines(left_con_pts, right_con_pts)
                break
            except IndexError:
                opt_plane = opt_plane - 1
                continue
                
        #If the ratio condition is not met, force the decrement to try again
        else:
            opt_plane = opt_plane - 1
            continue

    return p_sa_angle

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
axial_acpc_vol_aligned_name = config['axial_acpc_vol_aligned_name']
sagittal_acpc_vol_aligned_name = config['sagittal_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed p-SA values for each scan
p_sa_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
start_ax_plane_processing_params = {
    "brain_window_center": 40,
    "brain_window_width": 80,
    "csf_window_center": 15,
    "csf_window_width": 10,
    "third_ven_seg_initial_gauss_blur_size": 3,
    "msp_adaptive_thresh_size": 75,
    "third_ven_cen_roi_row_offset": 10,
    "third_ven_cen_roi_col_offset": 10,
    "third_ven_roi_major_axis": 35,
    "third_ven_roi_minor_axis": 15,
    "third_ven_roi_tilt_degrees": -75
}

In [ ]:
opt_ax_plane_processing_params = {
    "ax_adaptive_thresh_size": 75,
    "ax_lat_ven_roi_semi_major_axis": 50,
    "ax_lat_ven_roi_semi_minor_axis": 40,
    "lv_roi_close_kernel_size": 6,
    "num_close_iters_lv_roi_ax": 5,
    "max_superior_search_ax_planes": 15,
    "wm_low_thresh": 22,
    "wm_high_thresh": 28,
    "cc_roi_col_offset": 10, 
    "ax_plane_gauss_blur_kernel_size": 3, 
    "brain_window_center":40, 
    "brain_window_width":80
}

In [ ]:
occipital_horns_segment_params = {
    "lat_ven_seg_initial_gauss_blur_size": 3,
    "lv_roi_close_kernel_size": 6,
    "num_close_iters_roi_ax": 5,
    "ax_adaptive_thresh_size": 75,
    "ven_seg_to_full_brain_ratio": 0.1,
    "bilateral_filter_size": 11,
    "bilater_filter_sigma_color": 75,
    "bilater_filter_sigma_space": 75,
    "lv_middle_roi_col_offset": 5,
    "lv_middle_roi_row_offset": 20,
    "close_roi_left_prcnt_fbc": 0.2,
    "close_roi_cen_exclude": 0.1,
    "ax_lv_close_iter": 3,
    "ven_median_smooth_factor": 7,
    "min_ven_area_thresh": 20, 
    "brain_window_center":40, 
    "brain_window_width":80
}

##### SA pipeline

In [ ]:
for acc_num in accnum_list:
    
    pat = info_df[pat_id_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]  

    #Grab the PC coordinates to identify the mid sagittal plane on the AC-PC aligned sagittal image
    pca_str = info_df[pc_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]        
    [x,y,z] = pca_str.split(",")
    pca_x,pca_y,pca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]      
    pc = [pca_x, pca_y, pca_z]
    print(f'patient is {pat} and acc num is {acc_num}')

    
    #Aligned axial image for p-SA measurement
    ax_img_path = os.path.join(data_path,
                              acc_num,
                            axial_acpc_vol_aligned_name)

    #Aligned sagittal image for 3rd ventricle segmentation --> this will guide the axial slice selection
    sag_img_path = os.path.join(data_path,
                             acc_num,
                            axial_acpc_vol_aligned_name)

    #determine starting axial plane for evaluation
    start_ax_plane, msp_image, seg_area, mcs, mrs = determine_start_ax_plane_via_3v_seg(sag_img_path, ax_img_path, pc, 
                                                                                        start_ax_plane_processing_params)
     
    opt_plane = determine_opt_ax_plane_for_psa_via_wm_profile(start_ax_plane, ax_img_path, msp_image, seg_area, mcs, mrs,
                                                              opt_ax_plane_processing_params)

    #Segment occipital horns of the LVs on the determined optimal axial plane and fit lines between their medial walls
    
    p_sa_angle = segment_occipital_horns_fit_lines(ax_img_path, opt_plane, start_ax_plane, occipital_horns_segment_params)

        
    p_sa_df = pd.concat([p_sa_df, pd.DataFrame({pat_id_col_name:[pat],scan_id_col_name:["'" + acc_num + "'"], 
                                                   'p_SA':[p_sa_angle]
                                           })], ignore_index  = True)
        

In [ ]:
len(p_sa_df), sum(p_sa_df['p_SA'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(p_sa_df['p_SA'])
plt.xticks([x for x in range(10, 150, 10)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_p_sa_lim = 10
max_p_sa_lim = 140

p_sa_df[(p_sa_df['p_SA'] <= min_p_sa_lim) | (p_sa_df['p_SA'] >= max_p_sa_lim)]

In [ ]:
p_sa_df.to_csv(os.path.join(output_folder, 'p_sa.csv'))